# Load the raw telemetry file

In [4]:
import pandas as pd
df_scada = pd.read_json('raw_scada_logs.json')
print("=== 1. STRUCTURAL SUMMARY ===")
df_scada.info()

print("\n=== 2. STATISTICAL SPREAD ===")
print(df_scada.describe())

print("\n=== 3. STATUS CODE DISTRIBUTION ===")
print(df_scada['status_code'].value_counts())

=== 1. STRUCTURAL SUMMARY ===
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype        
---  ------          --------------  -----        
 0   timestamp       10000 non-null  datetime64[s]
 1   machine_id      10000 non-null  str          
 2   vibration_mm_s  9524 non-null   float64      
 3   spindle_temp    9511 non-null   float64      
 4   status_code     10000 non-null  str          
dtypes: datetime64[s](1), float64(2), str(2)
memory usage: 390.8 KB

=== 2. STATISTICAL SPREAD ===
                 timestamp  vibration_mm_s  spindle_temp
count                10000     9524.000000   9511.000000
mean   2026-06-15 17:34:27        4.492961     72.021813
min    2026-05-31 21:07:00       -0.131500     39.140000
25%    2026-06-08 04:59:53        3.685050     66.735000
50%    2026-06-15 16:43:28        4.485650     72.020000
75%    2026-06-23 08:24:16        5.292175     77.320000
max    2026-06-30 2

In [5]:
# --- BLOCK 2: SURGICAL CLEANING ---

# 1. Fix the Impossible Physics
# We use .loc[] to find rows where vibration < 0, and target specifically the 'vibration_mm_s' column to overwrite it with 0.0
print("Fixing negative calibration errors...")
df_scada.loc[df_scada['vibration_mm_s'] < 0, 'vibration_mm_s'] = 0.0

# 2. Impute Missing Sensor Data (Forward Fill)
# We fill the NaN gaps by carrying the last known good value forward.
print("Forward-filling missing sensor dropouts...")
df_scada['vibration_mm_s'] = df_scada['vibration_mm_s'].ffill()
df_scada['spindle_temp'] = df_scada['spindle_temp'].ffill()

# 3. Handle Edge Case (Backward Fill)
# If the VERY FIRST row in the dataset was a NaN, forward-fill won't work. 
# We run a quick backward-fill just in case to cover the start of the file.
df_scada['vibration_mm_s'] = df_scada['vibration_mm_s'].bfill()
df_scada['spindle_temp'] = df_scada['spindle_temp'].bfill()

# 4. Verify the Architecture
print("\n=== POST-CLEANING STRUCTURAL VERIFICATION ===")
df_scada.info()

print("\n=== POST-CLEANING STATISTICAL VERIFICATION ===")
print(df_scada.describe())

Fixing negative calibration errors...
Forward-filling missing sensor dropouts...

=== POST-CLEANING STRUCTURAL VERIFICATION ===
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype        
---  ------          --------------  -----        
 0   timestamp       10000 non-null  datetime64[s]
 1   machine_id      10000 non-null  str          
 2   vibration_mm_s  10000 non-null  float64      
 3   spindle_temp    10000 non-null  float64      
 4   status_code     10000 non-null  str          
dtypes: datetime64[s](1), float64(2), str(2)
memory usage: 390.8 KB

=== POST-CLEANING STATISTICAL VERIFICATION ===
                 timestamp  vibration_mm_s  spindle_temp
count                10000    10000.000000  10000.000000
mean   2026-06-15 17:34:27        4.492218     72.015379
min    2026-05-31 21:07:00        0.000000     39.140000
25%    2026-06-08 04:59:53        3.684575     66.700000
50%    2026-06-15 

## Block 3: Vectorized String Sanitization

**Objective:** Transform unstructured qualitative text (operator maintenance logs) into structured categorical data using Pandas vectorized `.str` accessors and Regular Expressions (Regex).

**Architectural Rationale:**
Python `for-loops` are computationally expensive and will bottleneck a pipeline when processing millions of rows. By utilizing the Pandas `.str` accessor, the operations are pushed down to the optimized C-level arrays, drastically reducing processing time and memory overhead.

**Execution Steps:**
1. **Data Integration:** Inject a simulated column of unstructured operator logs.
2. **Normalization:** Standardize all text casing.
3. **Regex Extraction:** Isolate specific alphanumeric error codes hidden within the text strings.
4. **Boolean Flagging:** Create automated alert flags based on specific keyword detection.

In [6]:
import numpy as np

# 1. DATA INTEGRATION: Injecting synthetic unstructured operator logs
np.random.seed(42)
raw_operator_notes = [
    "Routine check, nominal.",
    "Bearing noise detected, check err-404.",
    "Spindle overtemp. e-501 logged.",
    "Vibration anomaly, resolving E-909.",
    "Visual inspection passed."
]
df_scada['operator_log'] = np.random.choice(raw_operator_notes, size=len(df_scada))

print("--- RAW TEXT SAMPLES ---")
print(df_scada['operator_log'].head())

# 2. NORMALIZATION: Convert all text to lowercase to ensure uniform processing
df_scada['log_normalized'] = df_scada['operator_log'].str.lower()

# 3. REGEX EXTRACTION: Extract any string matching the pattern "e-" followed by 3 digits
df_scada['extracted_fault_code'] = df_scada['log_normalized'].str.extract(r'(e-\d{3})')

# 4. BOOLEAN FLAGGING: Create a True/False column if specific mechanical keywords are present
df_scada['mechanical_alert'] = df_scada['log_normalized'].str.contains('noise|vibration')

# 5. VERIFICATION: Display the newly structured dataframe columns
print("\n--- STRUCTURED STRING OUTPUT ---")
print(df_scada[['operator_log', 'extracted_fault_code', 'mechanical_alert']].head(10))

--- RAW TEXT SAMPLES ---
0    Vibration anomaly, resolving E-909.
1              Visual inspection passed.
2        Spindle overtemp. e-501 logged.
3              Visual inspection passed.
4              Visual inspection passed.
Name: operator_log, dtype: str

--- STRUCTURED STRING OUTPUT ---
                             operator_log extracted_fault_code  \
0     Vibration anomaly, resolving E-909.                e-909   
1               Visual inspection passed.                  NaN   
2         Spindle overtemp. e-501 logged.                e-501   
3               Visual inspection passed.                  NaN   
4               Visual inspection passed.                  NaN   
5  Bearing noise detected, check err-404.                  NaN   
6         Spindle overtemp. e-501 logged.                e-501   
7         Spindle overtemp. e-501 logged.                e-501   
8         Spindle overtemp. e-501 logged.                e-501   
9               Visual inspection passed.    

## Block 4: The Relational Bridge (Data Warehousing)

**Objective:** Fuse high-frequency production telemetry with static financial metadata using Relational Joins (`pd.merge`).

**Architectural Rationale:**
Data rarely exists in a single table. By utilizing `pd.merge()`, we replicate standard SQL `JOIN` behavior directly in-memory. 
* A `Left Join` ensures we retain 100% of our SCADA telemetry while pulling in matching financial data where available.
* This fusion allows us to perform cross-domain analytics, such as correlating spindle temperature with hourly operating costs.

In [9]:
import pandas as pd

# 1. SIMULATE ERP DATABASE EXTRACTION
# Creating a static financial metadata table for our 5 CNC machines
erp_data = {
    'machine_id': ['CNC-01', 'CNC-02', 'CNC-03', 'CNC-04', 'CNC-05'],
    'operating_cost_per_hr': [120.50, 120.50, 155.00, 155.00, 210.75],
    'maintenance_tier': ['Standard', 'Standard', 'Priority', 'Priority', 'Critical']
}
df_erp = pd.DataFrame(erp_data)

print("--- ERP FINANCIAL METADATA ---")
print(df_erp)

# 2. THE RELATIONAL MERGE (LEFT JOIN)
# We stitch the ERP data onto the SCADA logs using 'machine_id' as the shared key.
# 'how="left"' ensures we keep every single SCADA log, even if a machine isn't found in the ERP table.
df_master = pd.merge(df_scada, df_erp, on='machine_id', how='left')

# 3. VERIFICATION
print("\n--- MASTER DATAFRAME ARCHITECTURE (POST-MERGE) ---")
# Displaying specific columns to prove the fusion was successful
print(df_master[['timestamp', 'machine_id', 'spindle_temp', 'operating_cost_per_hr', 'maintenance_tier']].head())

# Check the new shape of our data
print(f"\nNew Master Shape: {df_master.shape[0]} rows, {df_master.shape[1]} columns")

--- ERP FINANCIAL METADATA ---
  machine_id  operating_cost_per_hr maintenance_tier
0     CNC-01                 120.50         Standard
1     CNC-02                 120.50         Standard
2     CNC-03                 155.00         Priority
3     CNC-04                 155.00         Priority
4     CNC-05                 210.75         Critical

--- MASTER DATAFRAME ARCHITECTURE (POST-MERGE) ---
            timestamp machine_id  spindle_temp  operating_cost_per_hr  \
0 2026-06-01 04:21:09     CNC-04         67.30                 155.00   
1 2026-06-07 14:43:32     CNC-05         71.16                 210.75   
2 2026-06-23 12:47:04     CNC-02         76.64                 120.50   
3 2026-06-11 13:02:30     CNC-05         67.82                 210.75   
4 2026-06-27 10:40:52     CNC-01         77.30                 120.50   

  maintenance_tier  
0         Priority  
1         Critical  
2         Standard  
3         Critical  
4         Standard  

New Master Shape: 10000 rows, 11 